# Transformer runs on Colab (XLM-R, SlavicBERT)

Fine-tunes the two transformers with `src/transformer/train_transformer.py` in every setting
(zero-shot, in-domain, mixed, and the mixed learning curve at 10 / 30 / 70 %).

**One-time setup (before the first session):** upload the local `data/processed/` folder
(`belarusian.jsonl`, `russian.jsonl`, `trasyanka.jsonl`, about 185 MB) to Google Drive as
`MyDrive/trasyanka/processed/`. It is gitignored, so the clone below does not include it.

**Every session:** Runtime → Change runtime type → **T4 GPU**, then run cells 1–4 in order.
Results go straight to `MyDrive/trasyanka/results/`, so a run that finished survives a disconnect.
After a disconnect, rerun cells 1–4 and cell 6 (which runs are done), then continue with the runs
that are missing.

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Drive, code and data

Mounts Drive, clones the repo (or pulls if it is already there), copies the processed data in,
and replaces `results/` with a link to `MyDrive/trasyanka/results/`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%%bash
set -e
DRIVE=/content/drive/MyDrive/trasyanka
REPO=/content/disentangling-trasyanka

if [ -d "$REPO" ]; then
    # Undo the results link first, so git pull can update results/ like any other folder
    if [ -L "$REPO/results" ]; then
        rm "$REPO/results"                      # removes only the link, not the Drive folder
        git -C "$REPO" checkout -- results
    fi
    git -C "$REPO" pull --ff-only
else
    git clone https://github.com/jlwise0111/disentangling-trasyanka.git "$REPO"
fi

mkdir -p "$REPO/data/processed"
cp "$DRIVE"/processed/*.jsonl "$REPO/data/processed/"

# results/ in the clone holds the files from git; the transformer runs write to Drive
# instead, so finished runs survive a disconnect
mkdir -p "$DRIVE/results"
rm -rf "$REPO/results"
ln -s "$DRIVE/results" "$REPO/results"

ls -lh "$REPO/data/processed"

In [ ]:
%cd /content/disentangling-trasyanka

## 3. Packages

Colab already has a GPU build of torch, numpy and scikit-learn. **Don't** install
`requirements.txt` here: it pins the CPU build of torch. Only the two missing packages are added
(`conllu` because `splits.py` imports the data-building code).

In [ ]:
!pip install -q conllu==6.0.0 transformers==5.24.1
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

## 4. Smoke test (about 2 minutes, saves nothing)

Checks the GPU, the model download (about 1 GB, cached for the session) and mixed precision on 200
sentences. The first line should end with `device cuda, mixed precision True`.

In [ ]:
!python -u src/transformer/train_transformer.py --model xlmr --setting in_domain --limit 200 --max-epochs 1

## 5. Real runs, one per cell

Each cell trains one run with early stopping on fam_sh and writes `results/<run>.json` and
`results/predictions/<run>.jsonl` to Drive when it finishes. Note the minutes per epoch from the
first run to estimate the rest. The order puts the main comparisons first.

### XLM-R

In [ ]:
!python -u src/transformer/train_transformer.py --model xlmr --setting in_domain

In [ ]:
!python -u src/transformer/train_transformer.py --model xlmr --setting mixed

In [ ]:
!python -u src/transformer/train_transformer.py --model xlmr --setting zero_shot

In [ ]:
!python -u src/transformer/train_transformer.py --model xlmr --setting mixed --fraction 0.1

In [ ]:
!python -u src/transformer/train_transformer.py --model xlmr --setting mixed --fraction 0.3

In [ ]:
!python -u src/transformer/train_transformer.py --model xlmr --setting mixed --fraction 0.7

### SlavicBERT

In [ ]:
!python -u src/transformer/train_transformer.py --model slavicbert --setting in_domain

In [ ]:
!python -u src/transformer/train_transformer.py --model slavicbert --setting mixed

In [ ]:
!python -u src/transformer/train_transformer.py --model slavicbert --setting zero_shot

In [ ]:
!python -u src/transformer/train_transformer.py --model slavicbert --setting mixed --fraction 0.1

In [ ]:
!python -u src/transformer/train_transformer.py --model slavicbert --setting mixed --fraction 0.3

In [ ]:
!python -u src/transformer/train_transformer.py --model slavicbert --setting mixed --fraction 0.7

### Shared spelling (zero-shot check)

Every source converted from its Latin tokens the same way, so the treebank-only letter cues
(и ъ = RU, і ў = BE) are gone. Compare with `xlmr` (0.350 on trasyanka_test).

In [ ]:
!python -u src/transformer/train_transformer.py --model xlmr --setting zero_shot --spelling shared

## 6. Which runs are done

Lists the transformer results on Drive. Download them from `MyDrive/trasyanka/results/` (both
the `.json` files and the `predictions/` folder) into the local `results/` folder, then run
`python src/analysis/compare_models.py` locally.

In [ ]:
import json
from pathlib import Path

expected = ["xlmr_in_domain", "xlmr_mixed", "xlmr", "xlmr_mixed_10", "xlmr_mixed_30", "xlmr_mixed_70",
            "slavicbert_in_domain", "slavicbert_mixed", "slavicbert", "slavicbert_mixed_10",
            "slavicbert_mixed_30", "slavicbert_mixed_70", "xlmr_shared"]
for name in expected:
    path = Path("results") / f"{name}.json"
    if path.exists():
        f1 = json.loads(path.read_text(encoding="utf-8"))["trasyanka_test"]["macro_f1"]
        print(f"done     {name:<22} trasyanka_test Macro-F1 {f1:.4f}")
    else:
        print(f"missing  {name}")